In [ ]:
%pip install numpy trimesh
import numpy as np
import trimesh
import matplotlib.pyplot as plt
from scipy.spatial import Delaunay
import tqdm as tqdm

In [ ]:
def franke(x,y):
  return 0.75*np.exp(-(9*x-2)**2/4-(9*y-2)**2/4) + 0.75*np.exp(-(9*x+1)**2/49-(9*y+1)/10) + 0.5*np.exp(-(9*x-7)**2/4-(9*y-3)**2/4) - 0.2*np.exp(-(9*x-4)**2-(9*y-7)**2)

def lap_franke(x,y):
  term1 = 243/16 * ((9*x-2)**2 + (9*y-2)**2-4) * np.exp(-(9*x-2)**2/4-(9*y-2)**2/4)
  term2 = 243/960400 * (400*(9*x+1)**2 - 7399) * np.exp(-(9*x+1)**2/49-(9*y+1)/10)
  term3 = 81/8 * ((9*x-7)**2 + (9*y-3)**2-4) * np.exp(-(9*x-7)**2/4-(9*y-3)**2/4)
  term4 = 324/5 * (1-(9*x-4)**2-(9*y-7)**2) * np.exp(-(9*x-4)**2-(9*y-7)**2)
  return term1 + term2 + term3 + term4

## Approximate function ($V$)

Suppose $U_i$ is our FEM value at vertex $p_i$

$$V(p_i)=U_i$$

However we don't have a prescribed value for a point that is not a vertex. Consider a point $p$ that is within the vertices $p_i , p_j, p_k$, we will find the value of $p$ with interpolation

Consider that $p$ partitions the triangle into three subtriangles $A_i, A_j, A_k$, where $A_m$ has vertex $p_m$ The interpolation will simply utilize the area of these triangles as weights for each vertex.

Since the interpolation should be in Barycentric Coordinates, we will have to normalize the triangle areas with the total area of triangle $p_i, p_j, p_k (A)$. Let $N_m = \frac{A_m}{A}$

$$V(p)=N_i U_i + N_j U_j + N_k U_k$$

In [ ]:
def find_triangle(point, division):

    x, y = point

    grid_size = int(round(2 / division))

    x_index = int(np.floor((x + 1) / division))
    y_index = int(np.floor((y + 1) / division))

    x_index = max(0, min(x_index, grid_size - 1))
    y_index = max(0, min(y_index, grid_size - 1))

    x_min = -1 + x_index * division
    x_max = min(x_min + division, 1)
    y_min = -1 + y_index * division
    y_max = min(y_min + division, 1)

    #print(f"Point: {point}, x_min: {x_min}, x_max: {x_max}, y_min: {y_min}, y_max: {y_max}")

    p_i = np.array([x_min, y_min])
    p_j = np.array([x_max, y_min])
    p_k = np.array([x_min, y_max])
    p_m = np.array([x_max, y_max])

    #print(f"Point: {point}, Triangle vertices: {p_i}, {p_j}, {p_k}, {p_m}")

    distances = [np.linalg.norm(point - p_i), np.linalg.norm(point - p_j), np.linalg.norm(point - p_k), np.linalg.norm(point - p_m)]

    max_distance = max(distances)
    max_index = distances.index(max_distance)

    if max_index == 0:
        return p_m, p_j, p_k
    elif max_index == 1:
        return p_i, p_k, p_m
    elif max_index == 2:
        return p_j, p_m, p_i
    else:
        return p_k, p_i, p_j
    

class FEM_V:
    def __init__(self,domain_path,division):
        self.domain = self.get_domain(domain_path)
        self.division = division

    def get_domain(self,domain_path):
        domain = {}
        with open(domain_path, 'r') as f:
            lines = f.readlines()
            lines.pop(0)  # Remove the header line
            for line in lines:
                x, y, u = map(float, line.strip().split(',')[:3])
                domain[(x, y)] = u
        return domain
    
    def get_value(self, point):
        if point in self.domain:
            return self.domain[point]
        else:
            p_i, p_j, p_k = find_triangle(point, self.division)
            triangles = [[point, p_j, p_k], [p_i, point, p_k], [p_i, p_j, point]]
            areas = [self.triangle_area(triangle) for triangle in triangles]
            total_area = sum(areas)
            areas = [area / total_area for area in areas]
            U = [self.domain[tuple(p)] for p in [p_i, p_j, p_k]]
            return sum(area * u for area, u in zip(areas, U))
        
    def triangle_area(self, triangle):
        p1, p2, p3 = triangle
        #cross product formula for area of triangle given by three points
        return 0.5 * abs((p2[0] - p1[0]) * (p3[1] - p1[1]) - (p3[0] - p1[0]) * (p2[1] - p1[1]))
    
    def __call__(self, point):
        return self.get_value(point)
    

V = FEM_V('./solutions/planar-quad-O1/0.csv', 1)
print(V((-1,-0.99)))

# Error Analysis 1

$$u(x,y)=(x^2-1)(y^2-1) \in \Gamma [-1,1] \times [-1,1]$$
$$-\Delta u = -2(x^2+y^2-2)$$
$$u(\Gamma_d) = 0 \Rightarrow g=0$$
$$E=|u-v|_{L^2(\Gamma)} \quad v \text{ is FEM Solution}$$

160,801 points from $[-1,1]\times[-1,1]$


|Refinement|Vertices| Time (s) |Error|
|:---:|:---:| :---:| :---:|
|0| 9 | 0.006393 |0.37484770759594555|
|1| 25 |0.005893 |0.121555231367444|
|2| 81| 0.021017|0.03255080912024765|
|3| 289|0.100850 |0.00827870388764948|
|4| 1089| 0.418767 |0.002078943950270176|
|5| 4225 |3.452945 |0.0005203245744961888|
|6| 16641 | 38.722023 |0001301205520557702|


$$u(x,y)=\sin(\pi x)\sin(\pi y) \in \Gamma [-1,1] \times [-1,1]$$
$$-\Delta u = 2\pi^2 \sin(\pi x) \sin(\pi y)$$
$$u(\Gamma_d) = 0 \Rightarrow g=0$$
$$E=|u-v|_{L^2(\Gamma)} \quad v \text{ is FEM Solution}$$

160,801 points from $[-1,1]\times[-1,1]$


|Refinement|Vertices| Time (s) |Error|
|:---:|:---:| :---:| :---:|
|0| 9 | 0.001265 |0.49875311720698207|
|1| 25 |0.004068  |0.3049129532336254|
|2| 81| 0.013029 |0.11388011679211257|
|3| 289|0.059195  |0.03177742194051499|
|4| 1089| 0.409578  |0.00817215639477807|
|5| 4225 |2.550736  |0.002057629490252998|
|6| 16641 | 35.123632  |0.0005153249632460317|

$$u(x,y)=\text{franke}\in \Gamma [-1,1] \times [-1,1]$$
$$-\Delta u = -\Delta \text{franke}$$
$$u(\Gamma_d) = \text{franke}(\Gamma_d)$$
$$E=|u-v|_{L^2(\Gamma)} \quad v \text{ is FEM Solution}$$

160,801 points from $[-1,1]\times[-1,1]$


|Refinement|Vertices| Time (s) |Error|
|:---:|:---:| :---:| :---:|
|0| 9 | 0.002035 |0.2330981282459142|
|1| 25 |0.008801  |0.24739898497866591|
|2| 81| 0.018982 |0.06686601496859687|
|3| 289|0.089436  |0.025046157246904652|
|4| 1089| 0.324690  |0.006841189742228643|
|5| 4225 |2.686268  |0.0017563828730377412|
|6| 16641 | 31.902233  |0.00044213424253047135|



In [ ]:
#%%script false --no-raise-error

def u(x,y):
  #return (x**2-1)*(y**2-1)
  #return np.sin(np.pi*x)*np.sin(np.pi*y)
  return franke(x,y)

division = [1, 0.5, 0.25, 0.125, 0.0625, 0.03125, 0.015625]

E=[]

for i in range(7):
    print("Mesh Refinement Level:", i)
    E_local = []
    V = FEM_V('./solutions/planar-franke-O1/'+str(i)+'.csv', division[i])
    with tqdm.tqdm(total=160801, desc=f"Computing L2 Error for Level {i}") as pbar:
        for x in np.arange(0, 1.0025, 0.0025):
            for y in np.arange(0, 1.0025, 0.0025):
                u_value = V((x, y))
                U_exact = u(x, y)
                E_local.append((u_value - U_exact)**2)
                #print(f"Point: ({x}, {y}), FEM Value: {u_value}, Exact Value: {U_exact}, Squared Error: {(u_value - U_exact)**2}")
                pbar.update(1)
    print("L2 Error:", np.sqrt(np.mean(E_local)))
    E.append(np.sqrt(np.mean(E_local)))

file = open('./solutions/planar-franke-O1/errors.csv','w')
file.write('Mesh Refinement Level,L2 Error\n')
for i in range(7):
    file.write(f'{i},{E[i]}\n')
file.close()

plt.figure()
plt.plot(range(0, 7), E, marker='o')
plt.title('L2 Error vs. Mesh Refinement')
plt.xlabel('Mesh Refinement Level')
plt.ylabel('L2 Error')
plt.xscale('linear')
plt.yscale('log')
plt.savefig('./solutions/planar-franke-O1/error_plot.png')
plt.show()



In [ ]:
vertices = [9, 25, 81, 289, 1089, 4225, 16641]
division = [1, 0.5, 0.25, 0.125, 0.0625, 0.03125, 0.015625]

for i in range(7):

    Vertices = []

    X,Y = np.meshgrid(np.linspace(0,1,3*2**i),np.linspace(0,1,3*2**i))

    print("Mesh Refinement Level:", i)
    u = []
    V = FEM_V('./solutions/planar-franke-O1/'+str(i)+'.csv', division[i])
    with tqdm.tqdm(total=160801, desc=f"creating points {i}") as pbar:
        for x in np.arange(0, 1.0025, 0.0025):
            for y in np.arange(0, 1.0025, 0.0025):
                if x in X and y in Y:
                    Vertices.append(V((x,y)))
                u_value = V((x, y))
                u.append(u_value)
                pbar.update(1)

    x = np.arange(0, 1.0025, 0.0025)
    y = np.arange(0, 1.0025, 0.0025)
    x, y = np.meshgrid(x, y)

    fig = plt.figure()
    ax = fig.add_subplot(projection='3d')

    # print(f"X shape: {X.shape}, Y shape: {Y.shape}, Vertices length: {len(Vertices)}, u length: {len(u)}")
    # print(f"X: {X}, Y: {Y}, Vertices: {Vertices[:5]}..., u: {u[:5]}...")  # Print first 5 elements for brevity

    surf = ax.plot_surface(x, y, np.array(u).reshape(x.shape), cmap='viridis')
    wire = ax.plot_wireframe(X, Y, np.array(Vertices).reshape(X.shape), color='black', linewidth=0.5)
    ax.set_title(f'FEM Solution Wireframe Plot - Vertices: {vertices[i]}')
    plt.savefig('./solutions/planar-franke-O1/surf_plot_'+str(i)+'.png')




# Approximate Function 2

In [ ]:
def round_key(p, ndigits=12):
    return (round(float(p[0]), ndigits), round(float(p[1]), ndigits))


def find_triangle(point, division):

    point = np.asarray(point, dtype=float)
    x, y = point

    grid_size = int(round(2 / division))

    x_index = int(np.floor((x + 1) / division))
    y_index = int(np.floor((y + 1) / division))

    x_index = max(0, min(x_index, grid_size - 1))
    y_index = max(0, min(y_index, grid_size - 1))

    x_min = -1 + x_index * division
    y_min = -1 + y_index * division
    x_max = x_min + division
    y_max = y_min + division

    p_i = np.array([x_min, y_min], dtype=float)  # bottom-left
    p_j = np.array([x_max, y_min], dtype=float)  # bottom-right
    p_k = np.array([x_min, y_max], dtype=float)  # top-left
    p_m = np.array([x_max, y_max], dtype=float)  # top-right

    # Local coordinates inside square cell
    sx = (x - x_min) / division
    sy = (y - y_min) / division

    # Same diagonal as your old farthest-corner method: p_j -- p_k
    if sx + sy <= 1.0:
        # lower-left triangle
        return p_i, p_j, p_k
    else:
        # upper-right triangle
        return p_m, p_k, p_j
    

def barycentric_coords(point, p1, p2, p3):

    point = np.asarray(point, dtype=float)
    p1 = np.asarray(p1, dtype=float)
    p2 = np.asarray(p2, dtype=float)
    p3 = np.asarray(p3, dtype=float)

    A = np.column_stack((p2 - p1, p3 - p1))
    xi_eta = np.linalg.solve(A, point - p1)

    xi = xi_eta[0]
    eta = xi_eta[1]

    L1 = 1.0 - xi - eta
    L2 = xi
    L3 = eta

    return L1, L2, L3


def p2_basis(L1, L2, L3):

    return np.array([
        L1 * (2 * L1 - 1),   # vertex i
        L2 * (2 * L2 - 1),   # vertex j
        L3 * (2 * L3 - 1),   # vertex k
        4 * L2 * L3,         # midpoint j-k
        4 * L1 * L3,         # midpoint i-k
        4 * L1 * L2          # midpoint i-j
    ])

class FEM_V2:
    def __init__(self, domain_path, division):
        self.domain = self.get_domain(domain_path)
        self.division = division

    def get_domain(self, domain_path):
        domain = {}

        with open(domain_path, 'r') as f:
            lines = f.readlines()

            # remove header if present
            if not lines[0][0].isdigit() and not lines[0].startswith("-"):
                lines.pop(0)

            for line in lines:
                parts = line.strip().split(',')
                if len(parts) < 3:
                    continue

                x = float(parts[0])
                y = float(parts[1])
                u = float(parts[2])

                domain[round_key((x, y))] = u

        return domain

    def node_value(self, p):
        key = round_key(p)

        if key not in self.domain:
            raise KeyError(
                f"Node {key} not found in CSV. "
                "This usually means you are using a P1 solution file, "
                "or the midpoint nodes were not written to the CSV."
            )

        return self.domain[key]

    def get_value(self, point):
        point = np.asarray(point, dtype=float)

        # If the point is exactly a stored FEM node, return it directly
        key = round_key(point)
        if key in self.domain:
            return self.domain[key]

        # Find physical triangle
        p_i, p_j, p_k = find_triangle(point, self.division)

        # P2 midpoint nodes, matching ordering [i, j, k, m_jk, m_ik, m_ij]
        m_jk = 0.5 * (p_j + p_k)
        m_ik = 0.5 * (p_i + p_k)
        m_ij = 0.5 * (p_i + p_j)

        nodes = [
            p_i,
            p_j,
            p_k,
            m_jk,
            m_ik,
            m_ij
        ]

        U = np.array([self.node_value(p) for p in nodes])

        # Barycentric coordinates relative to p_i, p_j, p_k
        L1, L2, L3 = barycentric_coords(point, p_i, p_j, p_k)

        phi = p2_basis(L1, L2, L3)

        return float(np.dot(U, phi))

    def __call__(self, point):
        return self.get_value(point)

# Error Analysis 2

$$u(x,y)=(x^2-1)(y^2-1) \in \Gamma [-1,1] \times [-1,1]$$
$$-\Delta u = -2(x^2+y^2-2)$$
$$u(\Gamma_d) = 0 \Rightarrow g=0$$
$$E=|u-v|_{L^2(\Gamma)} \quad v \text{ is FEM Solution}$$

160,801 points from $[-1,1]\times[-1,1]$

|Refinement|Vertices| Time (s) |Error|
|:---:|:---:| :---:| :---:|
|0| 25 |0.030046  |0.032215364582225896|
|1| 81| 0.076295|0.0040709292310741555|
|2| 289|0.129628 |0.0005088626337666907|
|3| 1089| 0.634545 |6.365379864461489e-05|
|4| 4225 |3.783874|7.950530765910383e-06|
|5| 16641 | 40.432681 |9.934197525017478e-07|

$$u(x,y)=\sin(\pi x)\sin(\pi y) \in \Gamma [-1,1] \times [-1,1]$$
$$-\Delta u = 2\pi^2 \sin(\pi x) \sin(\pi y)$$
$$u(\Gamma_d) = 0 \Rightarrow g=0$$
$$E=|u-v|_{L^2(\Gamma)} \quad v \text{ is FEM Solution}$$

160,801 points from $[-1,1]\times[-1,1]$

|Refinement|Vertices| Time (s) |Error|
|:---:|:---:| :---:| :---:|
|0| 25 |0.010218   |0.20287118776990454|
|1| 81| 0.022853 |0.03245517391636246|
|2| 289|0.089371  |0.004324465519334797|
|3| 1089| 0.416673  |0.0005486351236587917|
|4| 4225 |3.317555 |6.87337859181421e-05|
|5| 16641 | 39.039131  |8.595219388571188e-06|



$$u(x,y)=\sin(\pi x)\sin(\pi y) \in \Gamma [-1,1] \times [-1,1]$$
$$-\Delta u = 2\pi^2 \sin(\pi x) \sin(\pi y)$$
$$u(\Gamma_d) = 0 \Rightarrow g=0$$
$$E=|u-v|_{L^2(\Gamma)} \quad v \text{ is FEM Solution}$$

160,801 points from $[-1,1]\times[-1,1]$

|Refinement|Vertices| Time (s) |Error|
|:---:|:---:| :---:| :---:|
|0| 25 |0.013557   |0.20631237946056413|
|1| 81| 0.028949 |0.04898120064350327|
|2| 289|0.120025  |0.01585923259044396|
|3| 1089| 0.493018  |0.002133112062433032|
|4| 4225 |2.978242 |0.000297725822168739|
|5| 16641 | 38.413078  |3.805633226941439e-05|



In [ ]:
#%%script false --no-raise-error

def u(x,y):
  #return (x**2-1)*(y**2-1)
  #return np.sin(np.pi*x)*np.sin(np.pi*y)
  return franke(x,y)

division = [1, 0.5, 0.25, 0.125, 0.0625, 0.03125, 0.015625]

E=[]

for i in range(6):
    print("Mesh Refinement Level:", i)
    E_local = []
    V = FEM_V2(f'./solutions/planar-franke-O2/{i}.csv', division[i])
    with tqdm.tqdm(total=160801, desc=f"Computing L2 Error for Level {i}") as pbar:
        for x in np.arange(0, 1.0025, 0.0025):
            for y in np.arange(0, 1.0025, 0.0025):
                u_value = V((x, y))
                U_exact = u(x, y)
                E_local.append((u_value - U_exact)**2)
                #print(f"Point: ({x}, {y}), FEM Value: {u_value}, Exact Value: {U_exact}, Squared Error: {(u_value - U_exact)**2}")
                pbar.update(1)
    print("L2 Error:", np.sqrt(np.mean(E_local)))
    E.append(np.sqrt(np.mean(E_local)))

file = open('./solutions/planar-franke-O2/errors.csv','w')
file.write('Mesh Refinement Level,L2 Error\n')
for i in range(6):
    file.write(f'{i},{E[i]}\n')
file.close()

plt.figure()
plt.plot(range(0, 6), E, marker='o')
plt.title('L2 Error vs. Mesh Refinement')
plt.xlabel('Mesh Refinement Level')
plt.ylabel('L2 Error')
plt.xscale('linear')
plt.yscale('log')
plt.savefig('./solutions/planar-franke-O2/error_plot.png')
plt.show()


In [ ]:
vertices = [25, 81, 289, 1089, 4225, 16641]
division = [1, 0.5, 0.25, 0.125, 0.0625, 0.03125, 0.015625]
for i in range(6):
    print("Mesh Refinement Level:", i)
    u = []
    V = FEM_V2('./solutions/planar-franke-O2/'+str(i)+'.csv', division[i])
    with tqdm.tqdm(total=160801, desc=f"creating points {i}") as pbar:
        for x in np.arange(0, 1.0025, 0.0025):
            for y in np.arange(0, 1.0025, 0.0025):
                u_value = V((x, y))
                u.append(u_value)
                pbar.update(1)

    x = np.arange(0, 1.0025, 0.0025)
    y = np.arange(0, 1.0025, 0.0025)
    x, y = np.meshgrid(x, y)

    fig = plt.figure()
    ax = fig.add_subplot(projection='3d')

    surf = ax.plot_wireframe(x, y, np.array(u).reshape(x.shape), cmap='viridis')
    ax.set_title(f'FEM Solution Wireframe Plot - Vertices: {vertices[i]}')
    plt.savefig('./solutions/planar-franke-O2/wireframe_plot_'+str(i)+'.png')




# Approximate Function 3

In [ ]:
def q3_lagrange_basis_1d(t):

    return np.array([
        -9/16  * (t + 1/3) * (t - 1/3) * (t - 1),
         27/16 * (t + 1)   * (t - 1/3) * (t - 1),
        -27/16 * (t + 1)   * (t + 1/3) * (t - 1),
          9/16 * (t + 1)   * (t + 1/3) * (t - 1/3)
    ], dtype=float)


def q3_basis(xi, eta):

    Lxi = q3_lagrange_basis_1d(xi)
    Leta = q3_lagrange_basis_1d(eta)

    phi = np.zeros(16)

    for j in range(4):
        for i in range(4):
            A = i + 4*j
            phi[A] = Lxi[i] * Leta[j]

    return phi

def find_quad_cell(point, division):

    point = np.asarray(point, dtype=float)
    x, y = point

    grid_size = int(round(2 / division))

    x_index = int(np.floor((x + 1) / division))
    y_index = int(np.floor((y + 1) / division))

    # Clamp for points exactly on x=1 or y=1
    x_index = max(0, min(x_index, grid_size - 1))
    y_index = max(0, min(y_index, grid_size - 1))

    x_min = -1 + x_index * division
    y_min = -1 + y_index * division

    x_max = x_min + division
    y_max = y_min + division

    return x_min, x_max, y_min, y_max


def bicubic_cell_nodes(x_min, x_max, y_min, y_max):
  
    xs = np.linspace(x_min, x_max, 4)
    ys = np.linspace(y_min, y_max, 4)

    nodes = []

    for j in range(4):
        for i in range(4):
            nodes.append(np.array([xs[i], ys[j]], dtype=float))

    return nodes

def physical_to_reference(point, x_min, x_max, y_min, y_max):
    point = np.asarray(point, dtype=float)
    x, y = point

    xi = 2.0 * (x - x_min) / (x_max - x_min) - 1.0
    eta = 2.0 * (y - y_min) / (y_max - y_min) - 1.0

    return xi, eta


class FEM_Q3:
    def __init__(self, domain_path, division):

        self.domain = self.get_domain(domain_path)
        self.division = division

    def get_domain(self, domain_path):
        domain = {}

        with open(domain_path, 'r') as f:
            lines = f.readlines()

            if len(lines) == 0:
                return domain

            # Remove header if present
            first = lines[0].strip()
            if len(first) > 0:
                if not first[0].isdigit() and not first.startswith("-"):
                    lines.pop(0)

            for line in lines:
                parts = line.strip().split(',')

                if len(parts) < 3:
                    continue

                try:
                    x = float(parts[0])
                    y = float(parts[1])
                    u = float(parts[2])
                except ValueError:
                    continue

                domain[round_key((x, y))] = u

        return domain

    def node_value(self, p):
        key = round_key(p)

        if key not in self.domain:
            raise KeyError(
                f"Node {key} not found in CSV. "
                "For Q3 bicubic interpolation, the CSV must contain "
                "the 16-node bicubic mesh values, including the two "
                "edge nodes per edge and four interior nodes per quad."
            )

        return self.domain[key]

    def get_value(self, point):
        point = np.asarray(point, dtype=float)

        # If point is exactly a stored FEM node, return it directly
        key = round_key(point)
        if key in self.domain:
            return self.domain[key]

        # Find containing structured quad
        x_min, x_max, y_min, y_max = find_quad_cell(point, self.division)

        # Get 16 Q3 nodes in local order A = i + 4*j
        nodes = bicubic_cell_nodes(x_min, x_max, y_min, y_max)

        # Collect nodal solution values
        U = np.array([self.node_value(p) for p in nodes], dtype=float)

        # Map point to reference square [-1, 1]^2
        xi, eta = physical_to_reference(point, x_min, x_max, y_min, y_max)

        # Evaluate bicubic basis
        phi = q3_basis(xi, eta)

        return float(np.dot(U, phi))

    def __call__(self, point):
        return self.get_value(point)

# Error Analysis 3

$$u(x,y)=(x^2-1)(y^2-1) \in \Gamma [-1,1] \times [-1,1]$$
$$-\Delta u = -2(x^2+y^2-2)$$
$$u(\Gamma_d) = 0 \Rightarrow g=0$$
$$E=|u-v|_{L^2(\Gamma)} \quad v \text{ is FEM Solution}$$

160,801 points from $[-1,1]\times[-1,1]$


|Refinement| Vertices | Time (s) | Error|
|:---:|:---:| :---:| :---:|
|0|100|0.283240 |1.1717114624267517e-15 | 
|1|361| 1.246660| 3.872222683137446e-15|
|2|1369|4.292793| 6.904135575306066e-15|
|3|5329| 20.155206 |1.1508582370321206e-14 |
|4|21025|132.049323 |6.27627280566881e-14 |

$$u(x,y)=(x^2-1)(y^2-1) \in \Gamma [-1,1] \times [-1,1]$$
$$-\Delta u = -2(x^2+y^2-2)$$
$$u(\Gamma_d) = 0 \Rightarrow g=0$$
$$E=|u-v|_{L^2(\Gamma)} \quad v \text{ is FEM Solution}$$

160,801 points from $[-1,1]\times[-1,1]$


|Refinement| Vertices | Time (s) | Error|
|:---:|:---:| :---:| :---:|
|0|100|0.239949  |0.004152934544001213 | 
|1|361| 0.762704 | 0.0002750300187316374|
|2|1369|3.832329 | 1.749399064873429e-05|
|3|5329| 15.577666  |1.0983786087891423e-06 |
|4|21025|117.026044 |6.871912894043065e-08|


$$u(x,y)=(x^2-1)(y^2-1) \in \Gamma [-1,1] \times [-1,1]$$
$$-\Delta u = -2(x^2+y^2-2)$$
$$u(\Gamma_d) = 0 \Rightarrow g=0$$
$$E=|u-v|_{L^2(\Gamma)} \quad v \text{ is FEM Solution}$$

160,801 points from $[-1,1]\times[-1,1]$


|Refinement| Vertices | Time (s) | Error|
|:---:|:---:| :---:| :---:|
|0|100|0.307981  |0.050821889482754276 | 
|1|361| 0.896976 | 0.006718095585165243|
|2|1369|3.648907 | 0.0006836583062016555|
|3|5329| 16.178398  |4.742547602846163e-05 |
|4|21025|112.144778  |2.975765991701923e-06|


In [ ]:
def u(x,y):
  #return (x**2-1)*(y**2-1)
  #return np.sin(np.pi*x)*np.sin(np.pi*y)
  return franke(x,y)

nx_initial = 3
num_subdivisions = [0, 1, 2, 3, 4]  # Corresponding to mesh refinement levels 0 to 4

division = 2.0 / (nx_initial * 2**np.array(num_subdivisions))

print("Mesh Divisions:", division)

E=[]

for i in range(5):
    print("Mesh Refinement Level:", i)
    E_local = []
    V = FEM_Q3(f'./solutions/planar-franke-O3/{i}.csv', division[i])
    with tqdm.tqdm(total=160801, desc=f"Computing L2 Error for Level {i}") as pbar:
        for x in np.arange(0, 1.0025, 0.0025):
            for y in np.arange(0, 1.0025, 0.0025):
                u_value = V((x, y))
                U_exact = u(x, y)
                E_local.append((u_value - U_exact)**2)
                #print(f"Point: ({x}, {y}), FEM Value: {u_value}, Exact Value: {U_exact}, Squared Error: {(u_value - U_exact)**2}")
                pbar.update(1)
    print("L2 Error:", np.sqrt(np.mean(E_local)))
    E.append(np.sqrt(np.mean(E_local)))

file = open('./solutions/planar-franke-O3/errors.csv','w')
file.write('Mesh Refinement Level,L2 Error\n')
for i in range(5):
    file.write(f'{i},{E[i]}\n')
file.close()

plt.figure()
plt.plot(range(0, 5), E, marker='o')
plt.title('L2 Error vs. Mesh Refinement')
plt.xlabel('Mesh Refinement Level')
plt.ylabel('L2 Error')
plt.xscale('linear')
plt.yscale('log')
plt.savefig('./solutions/planar-franke-O3/error_plot.png')
plt.show()


In [ ]:
vertices = [100, 361, 1369, 5329, 21025]

nx_initial = 3
num_subdivisions = [0, 1, 2, 3, 4]  # Corresponding to mesh refinement levels 0 to 4

division = 2.0 / (nx_initial * 2**np.array(num_subdivisions))
for i in range(5):
    print("Mesh Refinement Level:", i)
    u = []
    V = FEM_Q3('./solutions/planar-franke-O3/'+str(i)+'.csv', division[i])
    with tqdm.tqdm(total=160801, desc=f"creating points {i}") as pbar:
        for x in np.arange(0, 1.0025, 0.0025):
            for y in np.arange(0, 1.0025, 0.0025):
                u_value = V((x, y))
                u.append(u_value)
                pbar.update(1)

    x = np.arange(0, 1.0025, 0.0025)
    y = np.arange(0, 1.0025, 0.0025)
    x, y = np.meshgrid(x, y)

    fig = plt.figure()
    ax = fig.add_subplot(projection='3d')

    surf = ax.plot_wireframe(x, y, np.array(u).reshape(x.shape), cmap='viridis')
    ax.set_title(f'FEM Solution Wireframe Plot - Vertices: {vertices[i]}')
    plt.savefig('./solutions/planar-franke-O3/wireframe_plot_'+str(i)+'.png')




# Error Analysis 4

$$u(x,y)=(x^2-1)(y^2-1) \in \Gamma [-1,1] \times [-1,1]$$
$$-\Delta u = -2(x^2+y^2-2)$$
$$u(\Gamma_d) = 0 \Rightarrow g=0$$
$$E=|u-v|_{L^2(\Gamma)} \quad v \text{ is FEM Solution}$$

160,801 points from $[-1,1]\times[-1,1]$

Epoch = 3000

|Refinement| Parameters | Time (s) | Error|
|:---:|:---:| :---:| :---:|
|0|177|20.459063 |0.005446517316575658 | 
|1|609| 23.552692| 0.01409726852166906|
|2|2241|29.928191| 0.001122827817224282|
|3|8577| 50.734495 |0.0011406475877935257 |
|4|33537|65.197837 |0.03604727001560827 |

Epoch = 10000

|Refinement| Parameters | Time (s) | Error|
|:---:|:---:| :---:| :---:|
|0|177|80.413943 |0.0045970410846571915 | 
|1|609| 86.030769| 0.000902438769491764|
|2|2241|103.819918| 0.002052274637022301|
|3|8577| 146.623238 |0.0016255439825724057 |
|4|33537|244.360521 |0.007149033341230471 |


Parameters = 2241, Epochs = 3000

|Refinement| Interior Points | Time (s) | Error|
|:---:|:---:| :---:| :---:|
|0|500|25.101050 |0.0010770910288547858 | 
|1|1000| 37.318173| 0.001406176404811609|
|2|2500|40.993354| 0.0011527703655555745|
|3|5000| 61.969624 |0.0013237619956418388 |
|4|10000|94.538452 |0.0020183243966297357 |
